In [1]:
%load_ext autoreload
%autoreload 3
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import sys, math, warnings
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

sys.path.append('/Users/blancamartin/Desktop/Voytek_Lab/spike_waveform/spikeparam/spikeparam')
sys.path.append('/Users/blancamartin/Desktop/Voytek_Lab/spike_waveform/')
sys.path.append('/Users/blancamartin/Desktop/Voytek_Lab/spike_waveform/spikeparam/AP_empirical_paper1/datasets/spe-1/spe1_helper_modules/')
sys.path.append('/Users/blancamartin/Desktop/Voytek_Lab/spike_waveform/spikeparam/AP_empirical_paper1/datasets/shared_helper_modules')

from ridge_regression_utils import (
    load_cell_data, load_hpf_lfp_windows, build_ridge_matrices,
    run_ridge_regression, apply_fdr, plot_ridge_results,
    WAVEFORM_LABELS,
)

warnings.filterwarnings('ignore', message='use_inf_as_na option is deprecated',
                        category=FutureWarning, module='seaborn')
warnings.filterwarnings('ignore', message=r'invalid value encountered in log10',
                        category=RuntimeWarning, module=r'.*specparam.*')

In [2]:
# ── Cell configuration ────────────────────────────────────────────────────────
cell_num = 13

# Symmetric 50 ms windows immediately around the spike
PRE_WIN      = (-0.055, -0.005)  # pre-spike:  −55 → −5 ms
POST_WIN     = ( 0.005,  0.055)  # post-spike: +5 → +55 ms
BASELINE_WIN = (-0.20,  -0.10)   # baseline: −200 → −100 ms

# HPF detrending for LFP amp/std targets (0.1 Hz removes slow electrode drift)
HPF_CUTOFF = 0.1   # Hz

# Ridge regression settings
ALPHAS          = np.logspace(-3, 3, 100)
N_PERM          = 1000
RNG_SEED        = 42
FORCE_RECOMPUTE = True   # True: rerun and overwrite pickle (needed after structure change)

In [3]:
# ── 1. Load data ──────────────────────────────────────────────────────────────
df_reg, specparam_by_spike, lfp_windows_by_spike = load_cell_data(cell_num)

# HPF-detrended LFP windows for amp/std targets
hpf_lfp = load_hpf_lfp_windows(cell_num, hpf_cutoff=HPF_CUTOFF)[:len(specparam_by_spike)]

Found 18 chunk files. Assembling master list...
Loading Chunks: 100%|██████████| 18/18 [00:02<00:00,  6.65it/s]
Success! Master list assembled with 3515 total spikes.
c13: 3515 spikes loaded


In [4]:
# ── 2. Build matrices ─────────────────────────────────────────────────────────
(X_waveform, X_log_isi, X_both, waveform_labels,
 Y, target_names, target_labels) = build_ridge_matrices(
    df_reg, specparam_by_spike, lfp_windows_by_spike,
    pre_win=PRE_WIN, post_win=POST_WIN, baseline_win=BASELINE_WIN,
    hpf_lfp_by_spike=hpf_lfp,
)

predictor_sets = {
    'Waveform only':      (X_waveform, waveform_labels),
    'Log ISI only':       (X_log_isi,  ['Log ISI']),
    'Waveform + Log ISI': (X_both,     waveform_labels + ['Log ISI']),
}

Target NaN %:
  Pre LFP Amp             0.0%
  Pre LFP Std             0.0%
  Pre Gamma AUC           0.0%
  Pre Exponent            0.0%
  Pre Theta AUC           0.0%
  Pre−BL LFP Amp          0.0%
  Pre−BL LFP Std          0.0%
  Pre−BL Gamma AUC        0.0%
  Pre−BL Exponent         0.0%
  Pre−BL Theta AUC        0.0%
  Post LFP Amp            0.0%
  Post LFP Std            0.0%
  Post Gamma AUC          0.0%
  Post Exponent           0.0%
  Post Theta AUC          0.0%
  Post−BL LFP Amp         0.0%
  Post−BL LFP Std         0.0%
  Post−BL Gamma AUC       0.0%
  Post−BL Exponent        0.0%
  Post−BL Theta AUC       0.0%
  Δ LFP Amp               0.0%
  Δ LFP Std               0.0%
  Δ Gamma AUC             0.0%
  Δ Exponent              0.0%
  Δ Theta AUC             0.0%


In [5]:
# ── 3. Ridge regression (5-fold CV + permutation test) ────────────────────────
import os, sys
sys.path.append('/Users/blancamartin/Desktop/Voytek_Lab/spike_waveform/spikeparam/AP_empirical_paper1/datasets/spe-1/spe1_helper_modules/')
from config import SPE1_PICKLE_ROOT

save_path = os.path.join(SPE1_PICKLE_ROOT, 'ridge_regression_pickles', f'c{cell_num}_ridge_results_hpf.pkl' if HPF_CUTOFF else f'c{cell_num}_ridge_results.pkl')

results = run_ridge_regression(
    Y, predictor_sets, target_names,
    n_perm=N_PERM, rng_seed=RNG_SEED, alphas=ALPHAS,
    save_path=save_path, force_recompute=FORCE_RECOMPUTE,
)

ridge CV: 100%|██████████| 75/75 [12:37<00:00, 10.10s/model, pred=Waveform +, target=delta_theta_]

  Saved: c13_ridge_results.pkl


In [6]:
# ── 4. FDR correction + save final results ────────────────────────────────────
# BH-FDR across all 75 tests (25 targets × 3 predictor sets) per cell.
# Valid under positive dependence (PRDS) — correlated targets satisfy this condition.
import pickle

results = apply_fdr(results, target_names, predictor_sets)

with open(save_path, 'wb') as f:
    pickle.dump(results, f)
print(f'Saved final results (with FDR): {os.path.basename(save_path)}')

FDR (BH, q=0.05): 21/75 raw p<0.05 → 11/75 after correction

Target                         Waveform only          Log ISI only    Waveform + Log ISI
----------------------------------------------------------------------------------------
pre_lfp_amp                 +0.0005~ α=1e+03      -0.0003~ α=1e+03      +0.0007~ α=1e+03
pre_lfp_std                 -0.0027  α=1e+03    +0.0083* α=1.1e+02    +0.0063* α=8.7e+02
pre_gamma_auc               -0.0042  α=1e+03    +0.0002~ α=8.7e+02      -0.0032  α=1e+03
pre_exponent                -0.0020  α=1e+03      +0.0003~ α=1e+03      -0.0010  α=1e+03
pre_theta_auc               -0.0028  α=1e+03      -0.0010  α=1e+03      -0.0029  α=1e+03
prebc_lfp_amp               -0.0022  α=1e+03         +0.0120* α=81    +0.0112* α=6.6e+02
prebc_lfp_std               -0.0014  α=1e+03      +0.0008* α=1e+03      -0.0003~ α=1e+03
prebc_gamma_auc             -0.0033  α=1e+03      -0.0033  α=1e+03      -0.0034  α=1e+03
prebc_exponent              -0.0018  α=1e+03     

## 2. Build Feature and Target Matrices

**Predictors (X)** — three competing sets:
- *Waveform only*: 8 spike shape features
- *Log ISI only*: 1 feature
- *Waveform + Log ISI*: 9 features combined

**Targets (Y)** — 25 LFP scalars (5 groups × 5 features):

| Group | Formula | Scientific question |
|-------|---------|-------------------|
| Pre absolute | mean(pre window) | LFP state when cell fires |
| Pre−BL | mean(pre) − mean(baseline) | LFP ramp into spike |
| Post absolute | mean(post window) | LFP state after spike |
| Post−BL | mean(post) − mean(baseline) | Spike-triggered response from baseline |
| Δ post−pre | mean(post) − mean(pre) | Net spike-triggered change |

Windows: Baseline −200→−100 ms  |  Pre −55→−5 ms  |  Post +5→+55 ms